In [ ]:
# 01 · STATE STAGE ACT 단일 정책 CONFIG · GitHub 중단 복구 · Drive 미사용
CFG = {
    # 저장 · ver2는 기존 DP 결과와 별도 Release에 저장
    'run_name': 'moveboxes_stage_chunk_deadline_v1',
    'profile': 'benchmark',
    'github_repository': 'SongYunu/moveBoxes',
    'project_dir': '/content/moveBoxes',
    'project_ref': '711dfd5caa71acab27aa663bec48b6bc97dcd183',
    'output_root': '/content/moveboxes_runs',

    # 데이터 / Colab 2026.07 · Python 3.12 · T4
    'repo_dir': '/content/berlin-marso-hackathon',
    'repo_url': 'https://github.com/marso-robotics/berlin-marso-hackathon.git',
    'repo_commit': '6048f33217f26ae39009a812f53c81171517f393',
    'data_dir': '/content/marso_data',
    'data_source': '/content/moveboxes_data_cache/marso_state_data.zip',
    'download_cache': '/content/moveboxes_data_cache',
    'packages': ['mani-skill==3.0.1', 'sapien==3.0.3', 'diffusers==0.38.0', 'hydra-core', 'omegaconf', 'gymnasium', 'tyro', 'h5py', 'kagglehub', 'tensorboard', 'matplotlib', 'transforms3d', 'imageio[ffmpeg]'],

    # 학습 · 시뮬레이터 없이 CPU 데이터 + GPU 모델
    'seed': 42,
    'num_demos': None,
    'batch_size': 64,
    'lr': 0.0001,
    'total_iters': {'easy': 12000, 'medium': 20000, 'hard': 30000},
    'amp': True,

    # 작은 State ACT · 기존 DP 체크포인트 사용 불가
    'history': 16,
    'chunk_size': 16,
    'width': 128,
    'heads': 4,
    'layers': 2,
    'latent_dim': 16,

    # 검증 / 중단 복구 / 작은 관측 위치 증강
    'save_freq': 1000,
    'warmup_steps': 500,
    'validation_batches': 8,
    'kl_weight': 0.001,
    'position_noise': 0.001,

    # 실행 · 매 스텝 재계획, 최근 XYZ 예측 평균, 집게는 최신 예측
    'temporal_decay': 0.25,
    'ensemble_window': 4,
    'ensemble_candidates': [1, 4],

    # 빠른 테스트 / 최종 평가 · 시드 분리, 기존 200스텝 유지
    'test_episodes': 8,
    'test_seed_start': 40000,
    'test_record_video': True,
    'tuning_episodes': 8,
    'tuning_seed_start': 20000,
    'benchmark_episodes': 100,
    'eval_seed_start': 30000,
    'max_episode_steps': {'easy': 200, 'medium': 200, 'hard': 200},
    'record_eval_video': True,

    # 출력
    'console_interval_seconds': 10,
    'team': 'my-team',

    # 실행 조건으로 행동 학습 · 빠른 테스트가 0이면 긴 평가 생략
    'action_training_mode': 'prior',
    'repair_iters': 2000,
    'allow_zero_success_evaluation': False,

    # 단계 판단 · 학습된 완료/복구 확신이 낮으면 현재 단계 유지
    'gate_threshold': 0.65,
    'stage_threshold': 0.6,
    'stage_loss_weight': 0.3,
    'gate_loss_weight': 0.3,

    # 복구 시연 · 수집 전용 expert, 학습/제출은 학습된 정책
    'recovery_episodes': 16,
    'recovery_max_attempts': 48,
    'recovery_seed_start': 100000,
    'collection_max_steps': {'easy': 500, 'medium': 900, 'hard': 1400},
    'noise_probability': 0.08,
    'action_noise_std': 0.12,
    'drop_probability': 0.015,

}


In [ ]:
# 02 · GitHub 코드 불러오기 (데이터·결과를 위해 Drive를 마운트하지 않습니다)
import importlib, os, subprocess, sys
from pathlib import Path

PROJECT = Path(CFG['project_dir'])
URL = 'https://github.com/'+CFG['github_repository']+'.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], cwd=PROJECT, text=True).strip()
    if remote != URL:
        raise RuntimeError('기존 프로젝트 폴더가 다른 저장소입니다. project_dir를 새 경로로 바꾸세요.')
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', CFG['project_ref']], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=PROJECT, check=True)
CFG['project_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
sys.path.insert(0, str(PROJECT))
# A fresh notebook run should not retain a previously imported project module.
for name in ('marso_experiment', 'marso_train_test', 'next_pick_sampling', 'next_pick_diagnostics',
             'marso_next_pick', 'github_store', 'github_data', 'colab_layout', 'build_modular_notebook',
             'colab_train_test_layout', 'build_train_test_notebook', 'colab_next_pick_layout',
             'build_next_pick_notebook', 'build_github_notebook', 'marso_github'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'))
for name in ('act_v2_model','act_v2_data','act_v2_policy','act_v2_eval','act_v2_experiment','build_act_v2_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'/'stages'))
for name in ('stage_schema','stage_model','stage_policy','stage_labels','stage_data','stage_teacher',
             'stage_collect','stage_eval','stage_experiment','stage_chunk_policy',
             'stage_pick_sampling','stage_pick_train','stage_all_pick_retrain',
             'stage_pick_finetune','stage_pick_diagnose','stage_reference_check',
             'build_stage_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
from stage_experiment import StageExperiment, source_bundle
experiment = StageExperiment(CFG, source_bundle())
print('사용 코드:', CFG['project_commit'])
print('코드 로드 완료. 03 셀로 연결하거나 다음 실행 셀에서 자동 연결합니다.')


In [ ]:
# 03 · GitHub 인증 / 저장된 결과 복원
# 기존 환경변수 GH_TOKEN → Colab 보안 비밀 → 입력창 순서로 인증합니다.
# Fine-grained token: SongYunu/moveBoxes → Contents: Read and write.
# 이 저장소는 공개이므로 여기에 올린 모델·로그·영상도 공개됩니다.
import os
# 개인 사본에서 직접 지정할 경우 아래 한 줄의 주석을 풀어 사용하세요.
# os.environ['GH_TOKEN'] = '본인 토큰'
experiment.connect()
experiment.show_results()


In [ ]:
# 04 · 새 런타임마다 환경 설치
experiment.install()


In [ ]:
# 05 · GitHub 데이터 다운로드·검증 / GPU와 정책 실행 확인
experiment.prepare_data()
experiment.check_runtime()


# 새 가중치로 재학습 · 모든 다음 상자 집기 보강

01~05의 GitHub 토큰·state 데이터·T4 설치 과정은 기존과 같습니다. 기존 run을 데이터 복구에 사용하고, 모델은 별도 `_all_pick_retrain_v1` run에서 무작위 초기화합니다. 현재 실패 모델과 과거 best는 초기 가중치로 사용하지 않습니다. 같은 새 run을 다시 열면 그 run의 optimizer/scaler/RNG checkpoint만 복구합니다.

성공했던 StageACT와 같은 history16/chunk16/width128/layers2, batch64, lr1e-4, prior 직접 학습을 사용합니다. 학습 때 설정된 act_horizon=1과 실행을 일치시켜 매 step XYZ를 재추론하고 temporal ensemble4를 적용하며, gripper는 최신 예측을 바로 적용합니다. 이전 pick2/carry6 실행과 gripper 확인 필터는 새 run에 적용하지 않습니다.

배치 50%는 첫 상자 및 이후 모든 상자 집기에서 상자 순서별 균등 표집합니다. 접근·정렬·하강·집기 전체가 포함되며 같은 상자 재집기는 새 상자 순서로 세지 않습니다. 나머지는 전체 시연·운반·놓기·전환·복구 표본입니다. 난이도마다 12,000 step, 500 step마다 전체 학습 상태를 GitHub에 백업합니다.

Easy 셀의 학습 → 공식 default 평가 → 영상을 먼저 확인하세요. loss는 성능 점수가 아닙니다. 각 셀은 다음 난이도에서도 같은 학습/정책 로직을 사용하며 후보 승자 선택이나 과거 가중치 복원은 하지 않습니다. Drive를 사용하지 않습니다.


In [ ]:
# 새 run 설정 · 기존 제출본 보존
import json, os, signal, shutil, subprocess, sys
from pathlib import Path
from IPython.display import Video,display
from stage_all_pick_retrain import prepare_retrain,train_retrain,package_retrain
RETRAIN_ITERS = 12000
PICK_FOCUS_FRACTION = .5
RETRAIN_SUFFIX = '_all_pick_retrain_v1'
MAX_STEPS = 200
UPSTREAM = Path(CFG['repo_dir'])
OFFICIAL_EVAL_CONFIG = UPSTREAM/'conf/eval/default.yaml'
finished = []
SMOKE_CONFIG = Path(CFG['output_root'])/(CFG['run_name']+RETRAIN_SUFFIX+'_smoke_eval.yaml')
SMOKE_CONFIG.write_text('eval:\n  n_episodes: 1\n  seeds: [61000]\n',encoding='utf-8')

def show_latest_video(level,label):
    folder = RUN_DIR/level/'integrated_official_eval'/label/'videos'
    videos = sorted(folder.rglob('*.mp4'),key=lambda path:path.stat().st_mtime)
    if not videos:
        raise FileNotFoundError(f'{folder}에 공식 평가 MP4가 없습니다.')
    display(Video(str(videos[-1]),embed=True,width=960))
print('기존 run (보존):',experiment.run_dir)
print('새 가중치 재학습:',CFG['run_name']+RETRAIN_SUFFIX)
def run_official(level, eval_config, label, candidate=None):
    candidate = Path(candidate) if candidate is not None else CANDIDATE
    output = RUN_DIR/level/'integrated_official_eval'/label
    output.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, str(UPSTREAM/'eval.py'), 'difficulty='+level,
        'obs_mode=state', 'policy=stage_policy:load_policy',
        'checkpoint='+str(candidate/'checkpoints'/level/'model.pt'),
        'eval_config='+str(eval_config), 'max_episode_steps='+str(MAX_STEPS),
        'hydra.run.dir='+str(output)]
    child_env = dict(os.environ)
    child_env['PYTHONPATH'] = str(candidate)+os.pathsep+str(UPSTREAM)+os.pathsep+child_env.get('PYTHONPATH','')
    log = output/'official_eval.log'
    with log.open('w', encoding='utf-8') as handle:
        process = subprocess.Popen(command, cwd=UPSTREAM, env=child_env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, errors='replace', bufsize=1)
        try:
            for line in process.stdout:
                handle.write(line)
                handle.flush()
                print(line, end='', flush=True)
            code = process.wait()
        finally:
            # Colab's stop button interrupts the kernel, not its GPU subprocess.
            if process.poll() is None:
                previous_sigint = signal.signal(signal.SIGINT, signal.SIG_IGN)
                try:
                    process.terminate()
                    try:
                        process.wait(timeout=5)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        process.wait()
                finally:
                    signal.signal(signal.SIGINT, previous_sigint)
            process.stdout.close()
    if code:
        raise RuntimeError(f'official eval failed ({level}); {log} 확인')
    return log


In [ ]:
# EASY 새 가중치 학습 · 같은 새 run에서는 전체 상태 복구
retrain = prepare_retrain(experiment,'easy',iterations=RETRAIN_ITERS,
    focus_fraction=PICK_FOCUS_FRACTION,run_suffix=RETRAIN_SUFFIX)
RUN_DIR = Path(retrain.run_dir)
train_retrain(retrain,'easy')
finished = [name for name in ('easy','medium','hard')
    if (retrain.run_dir/name/'training_complete.json').is_file()]
CANDIDATE = package_retrain(retrain,finished)
run_official('easy',SMOKE_CONFIG,'smoke')
show_latest_video('easy','smoke')
run_official('easy',OFFICIAL_EVAL_CONFIG,'default')
show_latest_video('easy','default')
print('새 run:',RUN_DIR)


In [ ]:
# MEDIUM 새 가중치 학습 · 같은 새 run에서는 전체 상태 복구
retrain = prepare_retrain(experiment,'medium',iterations=RETRAIN_ITERS,
    focus_fraction=PICK_FOCUS_FRACTION,run_suffix=RETRAIN_SUFFIX)
RUN_DIR = Path(retrain.run_dir)
train_retrain(retrain,'medium')
finished = [name for name in ('easy','medium','hard')
    if (retrain.run_dir/name/'training_complete.json').is_file()]
CANDIDATE = package_retrain(retrain,finished)
run_official('medium',SMOKE_CONFIG,'smoke')
show_latest_video('medium','smoke')
run_official('medium',OFFICIAL_EVAL_CONFIG,'default')
show_latest_video('medium','default')
print('새 run:',RUN_DIR)


In [ ]:
# HARD 새 가중치 학습 · 같은 새 run에서는 전체 상태 복구
retrain = prepare_retrain(experiment,'hard',iterations=RETRAIN_ITERS,
    focus_fraction=PICK_FOCUS_FRACTION,run_suffix=RETRAIN_SUFFIX)
RUN_DIR = Path(retrain.run_dir)
train_retrain(retrain,'hard')
finished = [name for name in ('easy','medium','hard')
    if (retrain.run_dir/name/'training_complete.json').is_file()]
CANDIDATE = package_retrain(retrain,finished)
run_official('hard',SMOKE_CONFIG,'smoke')
show_latest_video('hard','smoke')
run_official('hard',OFFICIAL_EVAL_CONFIG,'default')
show_latest_video('hard','default')
print('새 run:',RUN_DIR)


In [ ]:
# 새 학습 상태 전체 + 새 제출본 PC 다운로드
from google.colab import files
archive = shutil.make_archive(str(RUN_DIR.parent/(RUN_DIR.name+'_backup')),
    'zip',root_dir=RUN_DIR.parent,base_dir=RUN_DIR.name)
files.download(archive)
submission = shutil.make_archive(str(RUN_DIR/'retrained_candidate'),
    'zip',root_dir=CANDIDATE)
files.download(submission)
print('원래 제출본과 원래 checkpoint는 보존했습니다.')
